# Data Encryption in Backblaze B2

This notebook guides you through the different options for encrypting data in Backblaze B2, with Python examples that use the S3-compatible API through the AWS SDK for Python (boto3).

First, you will need to install the pinned dependencies from `requirements.txt`.


In [ ]:
%pip install -r requirements.txt

Restart your Jupyter kernel so that it can use the installed modules.

Load the standard Backblaze B2 environment variables and define a unique demo key prefix. Copy `.env.example` to `.env` and fill in your values before running this notebook.

Use a disposable demo bucket or prefix, not a production bucket root. Every object this notebook writes is scoped under a fresh `b2-data-encryption-demo/<run-id>/` prefix.


In [ ]:
import os
from uuid import uuid4
from dotenv import load_dotenv

load_dotenv('.env')

REQUIRED_ENV_VARS = (
    'B2_APPLICATION_KEY_ID',
    'B2_APPLICATION_KEY',
    'B2_BUCKET_NAME',
    'B2_REGION',
)

missing_env_vars = [name for name in REQUIRED_ENV_VARS if not os.environ.get(name)]
if missing_env_vars:
    raise RuntimeError(f"Missing required environment variable(s): {', '.join(missing_env_vars)}")

BUCKET_NAME = os.environ['B2_BUCKET_NAME']
B2_REGION = os.environ['B2_REGION']
# A local file that we will upload
FILENAME = 'photo.png'
DEMO_PREFIX = f'b2-data-encryption-demo/{uuid4().hex}/'
OBJECT_KEY = f'{DEMO_PREFIX}{FILENAME}'
SSE_B2_KEY = f'{DEMO_PREFIX}sse-b2-{FILENAME}'
DEFAULT_SSE_B2_KEY = f'{DEMO_PREFIX}sse-b2-default-{FILENAME}'
SSE_C_KEY = f'{DEMO_PREFIX}sse-c-{FILENAME}'
ENCRYPTED_KEY = f'{DEMO_PREFIX}encrypted-{FILENAME}'

print(f'Using B2 bucket: {BUCKET_NAME}')
print(f'Using B2 object prefix: {DEMO_PREFIX}')


## Create a boto3 Client

This example uses the Backblaze B2 S3-compatible API as the default access path. The authoritative configuration list is `.env.example`; the notebook validates the required variables before creating the client.

`B2_PUBLIC_URL_BASE` is optional and included in `.env.example` for consistency with other Backblaze B2 samples. This notebook does not read it because all operations use the S3-compatible API.


In [ ]:
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

client = boto3.client(
    's3',
    endpoint_url=f'https://s3.{B2_REGION}.backblazeb2.com',
    aws_access_key_id=os.environ['B2_APPLICATION_KEY_ID'],
    aws_secret_access_key=os.environ['B2_APPLICATION_KEY'],
    region_name=B2_REGION,
    config=Config(
        user_agent_extra='b2-data-encryption (backblaze-b2-samples)',
        connect_timeout=10,
        read_timeout=30,
        retries={'max_attempts': 3, 'mode': 'standard'},
    ),
)


## Writing Data to Backblaze B2 Without Explicit Encryption Settings

First, we'll check whether your configured bucket already has default encryption. If it does, Backblaze B2 may still return SSE-B2 headers for writes that do not request encryption explicitly.


In [ ]:
NO_BUCKET_ENCRYPTION_CODES = {
    'ServerSideEncryptionConfigurationNotFoundError',
    'NoSuchBucketEncryption',
    '404',
}

def get_bucket_encryption_configuration():
    try:
        response = client.get_bucket_encryption(Bucket=BUCKET_NAME)
        return response['ServerSideEncryptionConfiguration']
    except ClientError as e:
        error_code = e.response.get('Error', {}).get('Code')
        if error_code in NO_BUCKET_ENCRYPTION_CODES:
            return None
        raise

def restore_bucket_encryption(configuration):
    if configuration is None:
        try:
            client.delete_bucket_encryption(Bucket=BUCKET_NAME)
        except ClientError as e:
            error_code = e.response.get('Error', {}).get('Code')
            if error_code not in NO_BUCKET_ENCRYPTION_CODES:
                raise
    else:
        client.put_bucket_encryption(
            Bucket=BUCKET_NAME,
            ServerSideEncryptionConfiguration=configuration,
        )

bucket_encryption = get_bucket_encryption_configuration()
if bucket_encryption is None:
    print('Bucket default encryption is disabled.')
else:
    print('Bucket default encryption is enabled; writes without explicit encryption settings can still return SSE-B2 headers.')


Now we'll read a local file into memory. We'll upload this data with different encryption options throughout the remainder of the notebook. All object keys are created under `DEMO_PREFIX`, so concurrent runs do not reuse the same bucket-root keys.


In [ ]:
with open(FILENAME, 'rb') as f:
    # Read the entire file into memory
    plaintext_data = f.read()

    print(f'Read {len(plaintext_data)} bytes')

Writing data without encryption parameters sends no object-level encryption request. If your bucket default encryption is disabled, the response should have no encryption-related headers; if it is enabled, Backblaze B2 may add an SSE-B2 header.


In [ ]:
response = client.put_object(
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    Body=plaintext_data,
)

We can pretty-print the full response to inspect the encryption-related headers for your bucket configuration.


In [ ]:
from pprint import pprint

pprint(response)

Let's define a simple function to show just the encryption-related HTTP headers from the response.

In [ ]:
def print_encryption_headers(response):
    # This dict comprehension selects any headers that contain the substring 'encryption' in the key
    pprint({key: value for key, value in response['ResponseMetadata']['HTTPHeaders'].items() if 'encryption' in key})

Now it's easier to see whether Backblaze B2 returned encryption-related headers for this `put_object` response.


In [ ]:
print_encryption_headers(response)

## Writing Data to Backblaze B2 with SSE-B2

If we want to request SSE-B2 for a specific object, we specify the `ServerSideEncryption` argument.


In [ ]:
response = client.put_object(
    Bucket=BUCKET_NAME,
    Key=SSE_B2_KEY,
    Body=plaintext_data,
    ServerSideEncryption='AES256'
)

Note the server-side encryption header in the response

In [ ]:
print_encryption_headers(response)

When we read the data back, we don't need to specify any extra parameters

In [ ]:
response = client.get_object(
    Bucket=BUCKET_NAME,
    Key=SSE_B2_KEY,
)

Note the same server side encryption header in the `get_object` response:

In [ ]:
print_encryption_headers(response)

Compare the downloaded data to the plaintext we uploaded. No output means that the data is identical.

In [ ]:
downloaded_data = response['Body'].read()
assert downloaded_data == plaintext_data

Now let's temporarily enable default encryption for the demo bucket, write one object that relies on the bucket default, and restore the original bucket encryption setting in a `finally` block. This cell changes bucket-level configuration only while it is running.


In [ ]:
original_bucket_encryption = get_bucket_encryption_configuration()

try:
    response = client.put_bucket_encryption(
        Bucket=BUCKET_NAME,
        ServerSideEncryptionConfiguration={
            'Rules': [
                {
                    'ApplyServerSideEncryptionByDefault': {
                        'SSEAlgorithm': 'AES256'
                    }
                },
            ]
        }
    )
    pprint(response['ResponseMetadata']['HTTPStatusCode'])

    response = client.get_bucket_encryption(Bucket=BUCKET_NAME)
    pprint(response['ServerSideEncryptionConfiguration'])

    response = client.put_object(
        Bucket=BUCKET_NAME,
        Key=DEFAULT_SSE_B2_KEY,
        Body=plaintext_data,
    )
    print_encryption_headers(response)

    response = client.get_object(
        Bucket=BUCKET_NAME,
        Key=DEFAULT_SSE_B2_KEY,
    )
    print_encryption_headers(response)

    downloaded_data = response['Body'].read()
    assert downloaded_data == plaintext_data
finally:
    restore_bucket_encryption(original_bucket_encryption)


## Writing Data to Backblaze B2 with SSE-C

SSE-C uses AES-256, so we need to create a 256-bit (32-byte) key.

In [ ]:
from Crypto.Random import get_random_bytes

sse_c_key = get_random_bytes(32)

Here we just save the key to a file on disk. You would not do this in a real production system!

In [ ]:
with open('sse_c_keyfile', 'wb') as f:
    f.write(sse_c_key)

Now we can write the file using SSE-C. We must include the key in the `put_object` call as well as the `SSECustomerAlgorithm` argument.

In [ ]:
response = client.put_object(
    Bucket=BUCKET_NAME,
    Key=SSE_C_KEY,
    Body=plaintext_data,
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
    # No need to provide SSECustomerKeyMD5 - the SDK calculates it
)

Again, the response headers confirm the encryption mechanism. We can also see the MD5 hash of the encryption key.

In [ ]:
print_encryption_headers(response)

What happens if we try to read the file without the key?

In [ ]:
try:
    response = client.get_object(
        Bucket=BUCKET_NAME,
        Key=SSE_C_KEY
    )
except Exception as e:
    print(f'Exception: {e}')

What happens if we try to read the file with the wrong key?

In [ ]:
bad_key = get_random_bytes(32)
try:
    response = client.get_object(
        Bucket=BUCKET_NAME,
        Key=SSE_C_KEY,
        SSECustomerAlgorithm='AES256',
        SSECustomerKey=bad_key
    )
except Exception as e:
    print(f'Exception: {e}')

Let's read the file with the correct key

In [ ]:
response = client.get_object(
    Bucket=BUCKET_NAME,
    Key=SSE_C_KEY,
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
)
print_encryption_headers(response)

Again, the downloaded data should be identical to what we uploaded.

In [ ]:
downloaded_data = response['Body'].read()
assert downloaded_data == plaintext_data

## Client-side encryption using PyCryptodome

We must create a new key - we shouldn't use the one we already shared with B2!

In [ ]:
cse_key = get_random_bytes(32)

with open('cse_keyfile', 'wb') as f:
    f.write(cse_key)

For this example, we'll use AES Galois/Counter Mode (GCM) to encrypt data at the client. First, we create a new cipher object.

In [ ]:
from Crypto.Cipher import AES

cipher = AES.new(cse_key, AES.MODE_GCM)

Now we can encrypt the data. GCM provides data integrity by generating an authentication tag during encryption that verifies the data has not been tampered with.

In [ ]:
ciphertext, tag = cipher.encrypt_and_digest(plaintext_data)

We can verify that the ciphertext is at least different from the plaintext!

In [ ]:
assert ciphertext != plaintext_data

The nonce is a random value generated when we created the cipher. It adds randomness to the process, ensuring that encrypting the same plaintext will result in a different ciphertext.

In [ ]:
nonce = cipher.nonce

How big are these cryptographic objects?

In [ ]:
print(f"The plaintext data is {len(plaintext_data)} bytes")
print(f"The nonce is {len(nonce)} bytes; the ciphertext is {len(ciphertext)} bytes, and the tag is {len(tag)} bytes")

As you can see, the nonce and tag are 16 bytes, while the ciphertext is the same length as the plaintext data.

The convention is to concatenate the nonce, ciphertext and tag to form the encrypted data. The nonce is required to decrypt the ciphertext, and the tag allows us to verify that the data has not been tampered with in transit.

In [ ]:
encrypted_data = nonce + ciphertext + tag

Now we can upload the client-side encrypted data. No SSE-C parameters are required because the ciphertext is already encrypted before upload. If bucket default encryption is enabled, Backblaze B2 may still add an SSE-B2 response header on top of the client-side ciphertext.


In [ ]:
response = client.put_object(
    Bucket=BUCKET_NAME,
    Key=ENCRYPTED_KEY,
    Body=encrypted_data,
)
print_encryption_headers(response)

Download the encrypted data. Again, no special arguments are required.

In [ ]:
response = client.get_object(
    Bucket=BUCKET_NAME,
    Key=ENCRYPTED_KEY,
)

Extract the nonce, ciphertext and tag from the downloaded data

In [ ]:
downloaded_data = response['Body'].read()

# Python list slicing allows us to easily extract the different components from the downloaded data
downloaded_nonce = downloaded_data[:16]
downloaded_ciphertext = downloaded_data[16:-16]
downloaded_tag = downloaded_data[-16:]

Now we can decrypt the data and verify that it matches the plaintext

In [ ]:
cipher = AES.new(cse_key, AES.MODE_GCM, nonce=downloaded_nonce)

decrypted_data = cipher.decrypt_and_verify(downloaded_ciphertext, downloaded_tag)

assert decrypted_data == plaintext_data

## Encrypting Existing Data in Backblaze B2

To change the encryption mechanism for existing data to SSE-B2 or SSE-C, you can perform a server-side copy and provide the desired encryption settings.

First, we'll replace the current demo object version with an SSE-B2 encrypted copy.

In [ ]:
response = client.copy_object(
    CopySource={
        'Bucket': BUCKET_NAME,
        'Key': OBJECT_KEY
    },
    # Destination
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    ServerSideEncryption='AES256'
)
print_encryption_headers(response)

Remember, B2 buckets are versioned, so we need to delete the previous demo object version created by the copy operation.

The response from `copy_object` includes the version ID of the prior demo object version, so it's straightforward to delete that specific version. The 204 response indicates success.

In [ ]:
response = client.delete_object(
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    VersionId=response['CopySourceVersionId']
)
pprint(response['ResponseMetadata']['HTTPStatusCode'])

To replace the SSE-B2-encrypted file with SSE-C we need to provide the SSE-C key, as before.

In [ ]:
response = client.copy_object(
    CopySource={
        'Bucket': BUCKET_NAME,
        'Key': OBJECT_KEY
    },
    # Destination
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
    # No need to provide SSECustomerKeyMD5 - the SDK calculates it
)
print_encryption_headers(response)

Again, delete the prior demo object version created by the copy operation.

In [ ]:
response = client.delete_object(
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    VersionId=response['CopySourceVersionId']
)
pprint(response['ResponseMetadata']['HTTPStatusCode'])

To encrypt existing data with CSE, first we need to GET the plaintext

In [ ]:
source = client.get_object(
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    SSECustomerAlgorithm='AES256',
    SSECustomerKey=sse_c_key
)
print_encryption_headers(source)

Now we encrypt the plaintext and PUT the encrypted data. No SSE-C headers are required; if bucket default encryption is enabled, Backblaze B2 may still report SSE-B2 for the stored ciphertext.


In [ ]:
# Encrypt the downloaded data
cipher = AES.new(cse_key, AES.MODE_GCM)
ciphertext, tag = cipher.encrypt_and_digest(source['Body'].read())

encrypted_data = cipher.nonce + ciphertext + tag

# Upload the encrypted data
response = client.put_object(
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    Body=encrypted_data,
)
print_encryption_headers(response)

Once more, delete the prior demo object version for this run-specific key.

In [ ]:
response = client.delete_object(
    Bucket=BUCKET_NAME,
    Key=OBJECT_KEY,
    VersionId=source['VersionId']
)
pprint(response['ResponseMetadata']['HTTPStatusCode'])